# Fire Extinguisher Detection - Training Notebook
## YOLOv8s Fine-Tuning



## 1. Environment Setup

In [ ]:
!nvidia-smi
!pip install ultralytics -q

In [ ]:
from ultralytics import YOLO
import torch
import os
import yaml
import glob
import shutil
from collections import Counter
from IPython.display import Image as IPImage, display

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    print(f"VRAM: {vram / 1e9:.1f} GB")

## 2. Configuration

In [ ]:
# --- Paths ---
# IMPORTANT: Update DATASET_DIR to match your Kaggle dataset mount path
DATASET_DIR = "/kaggle/input/fire-extinguishers-yolov8/fire extinguishers.v1i.yolov8"
DATA_YAML = "/kaggle/working/data_fire_ext.yaml"
OUTPUT_DIR = "/kaggle/working/"

# --- Model ---
MODEL_NAME = "yolov8s.pt"
IMAGE_SIZE = 640
EPOCHS = 50
BATCH_SIZE = 16
PATIENCE = 15
SEED = 42

print(f"Model: {MODEL_NAME}")
print(f"Image size: {IMAGE_SIZE} | Epochs: {EPOCHS} | Batch: {BATCH_SIZE}")

## 3. Prepare Dataset Configuration

In [ ]:
with open(os.path.join(DATASET_DIR, "data.yaml"), 'r') as f:
    data_config = yaml.safe_load(f)

data_config['train'] = os.path.join(DATASET_DIR, "train", "images")
data_config['val'] = os.path.join(DATASET_DIR, "valid", "images")
data_config['test'] = os.path.join(DATASET_DIR, "test", "images")

with open(DATA_YAML, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("data.yaml:")
print(yaml.dump(data_config, default_flow_style=False))

## 4. Dataset Analysis

In [ ]:
names_raw = data_config.get('names', {})
class_names = {i: v for i, v in enumerate(names_raw)} if isinstance(names_raw, list) else names_raw
class_counts = Counter()
split_counts = {'train': Counter(), 'valid': Counter(), 'test': Counter()}

for split in ['train', 'valid', 'test']:
    label_dir = os.path.join(DATASET_DIR, split, 'labels')
    n_images = len(glob.glob(os.path.join(label_dir, '*.txt')))
    for label_file in glob.glob(os.path.join(label_dir, '*.txt')):
        with open(label_file, 'r') as f:
            for line in f:
                cid = int(line.strip().split()[0])
                class_counts[cid] += 1
                split_counts[split][cid] += 1
    print(f"{split}: {n_images} images")

print(f"\nTotal annotations: {sum(class_counts.values())}")
for cid, count in sorted(class_counts.items()):
    print(f"  {class_names.get(cid, cid)}: {count}")

## 5. Train Model

In [ ]:
model = YOLO(MODEL_NAME)

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    workers=4,
    seed=SEED,

    # Optimizer
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    warmup_bias_lr=0.01,

    # Simplified augmentation
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.0,
    copy_paste=0.0,
    erasing=0.0,

    # Strategy
    close_mosaic=10,
    amp=True,

    # Output
    project=OUTPUT_DIR,
    name="fire_extinguisher_training",
    exist_ok=True,
    verbose=True,
)

## 6. Training Results

In [ ]:
train_dir = os.path.join(OUTPUT_DIR, "fire_extinguisher_training")

plots = [
    'results.png',
    'confusion_matrix.png',
    'confusion_matrix_normalized.png',
    'P_curve.png',
    'R_curve.png',
    'PR_curve.png',
    'F1_curve.png',
]

for name in plots:
    path = os.path.join(train_dir, name)
    if os.path.exists(path):
        print(f"\n{'='*60}")
        print(f"  {name}")
        print(f"{'='*60}")
        display(IPImage(filename=path, width=800))

## 7. Test Set Validation

In [ ]:
BEST_WEIGHTS = os.path.join(OUTPUT_DIR, "fire_extinguisher_training/weights/best.pt")
best_model = YOLO(BEST_WEIGHTS)

test_results = best_model.val(
    data=DATA_YAML,
    split='test',
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    conf=0.25,
    iou=0.6,
    verbose=True,
)

print(f"\n{'='*40}")
print(f"  TEST SET RESULTS")
print(f"{'='*40}")
print(f"  mAP50:      {test_results.box.map50:.4f}")
print(f"  mAP50-95:   {test_results.box.map:.4f}")
print(f"  Precision:  {test_results.box.mp:.4f}")
print(f"  Recall:     {test_results.box.mr:.4f}")

for i, name in test_results.names.items():
    print(f"  {name} AP50: {test_results.box.ap50[i]:.4f}")

## 8. Export Model

In [ ]:
# Copy best weights for easy download
dest = os.path.join(OUTPUT_DIR, "fire_extinguisher_best.pt")
shutil.copy(BEST_WEIGHTS, dest)
print(f"Best weights saved to {dest}")
print(f"Size: {os.path.getsize(dest) / 1e6:.1f} MB")

## 9. Output Files

In [ ]:
print("Output files:")
print("-" * 60)
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in files:
        filepath = os.path.join(root, f)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        if size_mb > 0.1:
            print(f"  {filepath} ({size_mb:.1f} MB)")